In [1]:
from dataclasses import asdict
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
from tqdm import tqdm

from config import PoseConfig, FeatureConfig, TrackConfig
from step2_pose import PoseEstimator
from tracking import Tracker

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/Users/avgreensoup/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ModuleNotFoundError: No module named 'filterpy'

In [2]:
def read_video_frames(video_path: Path) -> List[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    frames: List[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()
    return frames

In [3]:
def generate_windows(num_frames: int, seq_len: int, stride: int) -> List[Tuple[int, int]]:
    if num_frames == 0:
        return []
    if num_frames <= seq_len:
        return [(0, num_frames)]
    windows: List[Tuple[int, int]] = []
    for start in range(0, num_frames - seq_len + 1, stride):
        end = start + seq_len
        windows.append((start, end))
    if not windows:
        windows.append((num_frames - seq_len, num_frames))
    return windows

In [4]:
import numpy as np

def window_skeleton(
    frames: list[np.ndarray],
    pose: PoseEstimator,
    track_cfg: TrackConfig,
    seq_len: int,
    max_persons: int = 1,
) -> np.ndarray:
    tracker = Tracker(
        max_age=track_cfg.max_age,
        min_hits=track_cfg.min_hits,
        iou_threshold=track_cfg.iou_threshold,
    )

    # Collect keypoints per track id
    keypoints_by_track: dict[int, list[np.ndarray]] = {}
    detections_per_frame = pose.infer_batch(frames)
    for detections in detections_per_frame:
        detections = sorted(detections, key=lambda d: d[1], reverse=True)[:max_persons]
        tracks = tracker.step(detections)
        for track_id, _, keypoints in tracks:
            keypoints_by_track.setdefault(track_id, []).append(keypoints)

    if not keypoints_by_track:
        # No people detected; return zeros
        return np.zeros((seq_len, 17, 3), dtype=np.float32)

    # Pick the longest / highest-confidence track
    def track_score(seq):
        conf = [kp[:, 2].mean() for kp in seq]
        return len(seq), float(np.mean(conf))

    best_track = max(keypoints_by_track.values(), key=track_score)

    # Pad or truncate to seq_len frames
    if len(best_track) >= seq_len:
        best_track = best_track[:seq_len]
    else:
        pad = [np.zeros_like(best_track[0]) for _ in range(seq_len - len(best_track))]
        best_track = best_track + pad

    skeleton = np.stack(best_track, axis=0).astype(np.float32)  # (T, 17, 3)

    H, W = frames[0].shape[:2]
    skeleton = np.stack(best_track, axis=0).astype(np.float32)  # (T, 17, 3)

    # normalize xy to [0,1]
    skeleton[..., 0] /= W
    skeleton[..., 1] /= H

    # center on hip midpoint (COCO indices 11/12)
    hip_center = (skeleton[:, [11, 12], :2].mean(axis=1, keepdims=True))
    skeleton[..., :2] -= hip_center

    # optional scale normalization
    shoulder_dist = np.linalg.norm(skeleton[:, 5, :2] - skeleton[:, 6, :2], axis=-1, keepdims=True)
    scale = np.maximum(shoulder_dist, 1e-6)  # avoid divide by zero
    skeleton[..., :2] /= scale[:, None, :]
    return skeleton


In [5]:
def process_video(
    video_path: Path,
    label: int,
    pose: PoseEstimator,
    feature_cfg: FeatureConfig,
    track_cfg: TrackConfig,
    max_persons: int,
) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    frames = read_video_frames(video_path)
    windows = generate_windows(len(frames), feature_cfg.seq_len, feature_cfg.stride)
    skeletons: list[np.ndarray] = []
    for start, end in windows:
        window_frames = frames[start:end]
        skeletons.append(
            window_skeleton(
                frames=window_frames,
                pose=pose,
                track_cfg=track_cfg,
                seq_len=feature_cfg.seq_len,
                max_persons=max_persons,
            )
        )
    if not skeletons:
        skeletons.append(np.zeros((feature_cfg.seq_len, 17, 3), dtype=np.float32))

    skeleton_array = np.stack(skeletons, axis=0)  # (num_windows, T, 17, 3)
    labels = np.full((skeleton_array.shape[0],), label, dtype=np.int64)
    stats = {
        "num_frames": len(frames),
        "num_windows": skeleton_array.shape[0],
    }
    return skeleton_array, labels, stats

In [6]:
from pathlib import Path
import numpy as np
from tqdm import tqdm

pose_cfg = PoseConfig()
track_cfg = TrackConfig()
feature_cfg = FeatureConfig()
pose = PoseEstimator(
    weights=pose_cfg.weights,
    imgsz=pose_cfg.imgsz,
    conf=pose_cfg.conf,
    iou=pose_cfg.iou,
    device=pose_cfg.device,
    max_det=pose_cfg.max_det,
)

data_root = Path("RWF-2000_data")
output_root = Path("precomputed_graphs")
output_root.mkdir(parents=True, exist_ok=True)

label_pairs = [("Fight", 1), ("NonFight", 0)]
splits = ["train", "val"]

manifest = []
for split in splits:
    for label_name, label in label_pairs:
        src_dir = data_root / split / label_name
        if not src_dir.exists():
            continue
        dst_dir = output_root / split / label_name
        dst_dir.mkdir(parents=True, exist_ok=True)

        videos = sorted(list(src_dir.glob("*.mp4")) + list(src_dir.glob("*.avi")))
        for video_path in tqdm(videos, desc=f"{split}/{label_name}", unit="video"):
            out_path = dst_dir / f"{video_path.stem}.npz"
            skeletons, labels, stats = process_video(
                video_path=video_path,
                label=label,
                pose=pose,
                feature_cfg=feature_cfg,
                track_cfg=track_cfg,
                max_persons=1,  # keep top track; adjust if needed
            )
            np.savez_compressed(out_path, skeletons=skeletons, labels=labels)
            manifest.append({
                "split": split,
                "label_name": label_name,
                "label": label,
                "video": video_path.name,
                "num_frames": stats["num_frames"],
                "num_windows": stats["num_windows"],
                "output": str(out_path.relative_to(output_root)),
            })

# Optional: write metadata
import json
(output_root / "manifest.json").write_text(json.dumps({
    "pose_cfg": asdict(pose_cfg),
    "feature_cfg": asdict(feature_cfg),
    "track_cfg": asdict(track_cfg),
    "manifest": manifest,
}, indent=2))
print("Done:", len(manifest), "videos")


train/Fight:   0%|          | 2/800 [00:33<3:43:36, 16.81s/video]


KeyboardInterrupt: 